#### 📊 Data Preprocessing & Feature Engineering Project

### 🩺 Advanced Data Cleansing & Quality Optimization for Healthcare Analytics 

### 🧠 Objective
The main goal of this project is to handle complex missing values and treat clinical outliers in a healthcare dataset using multiple statistics and machine learning techniques, making the data model-ready.

### 📂 Dataset Overview
The dataset contains 400 patient records with the following attributes:
* **Demographics:** Age, Gender, Region
* **Medical Data:** BMI, Blood Pressure, Cholesterol, Glucose
* **Target Variable:** Disease Risk (0 or 1)

---

### 🛠️ Step 1: Library Ingestion & Framework Initialization

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats.mstats import winsorize

### 📂 Step 2: Dataset Ingestion & Initial Stream Configuration

In [2]:
df = pd.read_csv('../Datasets/patient_health_records.csv')

In [3]:
df

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,PID_1001,69.0,NaN,South,20.03,173.75,183.83,95.74,0
1,PID_1002,32.0,Male,East,22.74,105.61,195.74,82.85,0
2,PID_1003,78.0,Female,East,NaN,119.06,NaN,93.73,0
3,PID_1004,38.0,Female,South,32.77,106.86,76.18,99.88,0
4,PID_1005,41.0,Female,West,28.54,123.58,NaN,80.71,0
...,...,...,...,...,...,...,...,...,...
395,PID_1396,19.0,Female,North,30.38,130.62,208.24,86.77,1
396,PID_1397,52.0,Male,West,25.40,125.00,346.16,NaN,0
397,PID_1398,NaN,Female,East,19.97,131.33,198.93,96.90,1
398,PID_1399,43.0,Female,North,29.78,125.35,214.57,85.73,1


### 🔍 Step 3: Data Preview

In [4]:
df.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,PID_1001,69.0,NaN,South,20.03,173.75,183.83,95.74,0
1,PID_1002,32.0,Male,East,22.74,105.61,195.74,82.85,0
2,PID_1003,78.0,Female,East,NaN,119.06,NaN,93.73,0
3,PID_1004,38.0,Female,South,32.77,106.86,76.18,99.88,0
4,PID_1005,41.0,Female,West,28.54,123.58,NaN,80.71,0


In [5]:
df.tail()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
395,PID_1396,19.0,Female,North,30.38,130.62,208.24,86.77,1
396,PID_1397,52.0,Male,West,25.40,125.00,346.16,NaN,0
397,PID_1398,NaN,Female,East,19.97,131.33,198.93,96.90,1
398,PID_1399,43.0,Female,North,29.78,125.35,214.57,85.73,1
399,PID_1400,51.0,Female,South,27.49,118.89,161.13,76.92,0


In [6]:
df.shape

(400, 9)

### 🔍 Step 4: Missing Value Count

In [7]:
print(df.isnull().sum())

patient_id         0
age               40
gender            32
region            48
bmi               32
blood_pressure     0
cholesterol       40
glucose           35
disease_risk       0
dtype: int64


In [ ]:
missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_report = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percentage
})
print(missing_report)

                Missing Count  Percentage (%)
patient_id                  0            0.00
age                        40           10.00
gender                     32            8.00
region                     48           12.00
bmi                        32            8.00
blood_pressure              0            0.00
cholesterol                40           10.00
glucose                    35            8.75
disease_risk                0            0.00


### 🧮 Step 5: Mean Imputation

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='mean')

df_clean = df.copy()

df_clean['bmi'] = num_imputer.fit_transform(df_clean[['bmi']])

print("BMI missing values after imputation:", df_clean['bmi'].isnull().sum())

BMI missing values after imputation: 0


### 🧮 Step 6: Mode Imputation (Categorical)

In [ ]:
cat_imputer = SimpleImputer(strategy='most_frequent')

df_clean[['region', 'gender']] = cat_imputer.fit_transform(df_clean[['region', 'gender']])

print("Region missing values:", df_clean['region'].isnull().sum())
print("Gender missing values:", df_clean['gender'].isnull().sum())

Region missing values: 0
Gender missing values: 0


### 🎲 Step 7: Missing Indicator & Random Sample Imputation

In [ ]:
df_clean['age_isna'] = df_clean['age'].isnull().astype(int)

random_samples = df['age'].dropna().sample(df['age'].isnull().sum(), random_state=42)

random_samples.index = df[df['age'].isnull()].index

df_clean.loc[df['age'].isnull(), 'age'] = random_samples

print("Age missing values after random imputation:", df_clean['age'].isnull().sum())

Age missing values after random imputation: 0


### 🤖 Step 8: KNN Multivariate Imputation

In [12]:
from sklearn.impute import SimpleImputer , KNNImputer

knn_imputer = KNNImputer(n_neighbors=3)

num_cols = ['age', 'bmi', 'blood_pressure', 'cholesterol', 'glucose']

df_clean[num_cols] = knn_imputer.fit_transform(df_clean[num_cols])

print("Cholesterol missing values after KNN:", df_clean['cholesterol'].isnull().sum())

Cholesterol missing values after KNN: 0


### ⛓️ Step 9: MICE Algorithm Multivariate Imputation & Final Check

In [13]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_imputer = IterativeImputer(max_iter=10, random_state=42)

df_clean[num_cols] = mice_imputer.fit_transform(df_clean[num_cols])

print("Glucose missing values after MICE:", df_clean['glucose'].isnull().sum())
print("\n--- Final Check for Part A ---")
print(df_clean.isnull().sum())

Glucose missing values after MICE: 0

--- Final Check for Part A ---
patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
age_isna          0
dtype: int64


### 📏 Step 10: Outlier Detection & Removal via Z-Score Method

In [14]:
df_z_clean = df_clean.copy()

z_chol = np.abs(stats.zscore(df_z_clean['cholesterol']))
z_gluc = np.abs(stats.zscore(df_z_clean['glucose']))

outliers_chol = z_chol > 3
outliers_gluc = z_gluc > 3

print(f"Total outliers detected in Cholesterol: {outliers_chol.sum()}")
print(f"Total outliers detected in Glucose: {outliers_gluc.sum()}")

df_z_clean = df_z_clean[(z_chol <= 3) & (z_gluc <= 3)]
print(f"New dataset shape after Z-score removal: {df_z_clean.shape}")

Total outliers detected in Cholesterol: 16
Total outliers detected in Glucose: 16
New dataset shape after Z-score removal: (368, 10)


### 📦 Step 11: Outlier Detection & Trimming via IQR Method

In [15]:
Q1 = df_clean['bmi'].quantile(0.25)
Q3 = df_clean['bmi'].quantile(0.75)
IQR = Q3 - Q1

lower_line = Q1 - 1.5 * IQR
upper_line = Q3 + 1.5 * IQR

bmi_outliers = (df_clean['bmi'] < lower_line) | (df_clean['bmi'] > upper_line)
print(f"Total outliers detected in BMI using IQR: {bmi_outliers.sum()}")

df_iqr_clean = df_clean[~bmi_outliers]
print(f"New dataset shape after IQR removal: {df_iqr_clean.shape}")

Total outliers detected in BMI using IQR: 19
New dataset shape after IQR removal: (381, 10)


### ✂️ Step 12: Outlier Capping via Percentile Method (Winsorization)

In [16]:
lower_p = df_clean['bmi'].quantile(0.01)
upper_p = df_clean['bmi'].quantile(0.99)

df_percentile = df_clean.copy()

df_percentile['bmi'] = np.clip(df_percentile['bmi'], lower_p, upper_p)
print("Percentile capping completed successfully!")

Percentile capping completed successfully!


### 🪵 Step 13: Multivariate Outlier Winsorization

In [17]:
df_winsorized = df_clean.copy()

num_cols = ['blood_pressure', 'bmi', 'cholesterol', 'glucose']

for col in num_cols:
    df_winsorized[col] = winsorize(df_winsorized[col], limits=[0.05, 0.05])

print("Winsorization completed successfully on all columns!")

Winsorization completed successfully on all columns!


### 💾 Step 14: Final Cleansed Data Export & Target Storage

In [18]:
final_filename = 'final_cleaned_patient_records.csv'

df_winsorized.to_csv(final_filename, index=False)

print(f"Success! Your fully cleaned dataset is saved as: {final_filename}")
print(f"Final shape of the clean data: {df_winsorized.shape}")

Success! Your fully cleaned dataset is saved as: final_cleaned_patient_records.csv
Final shape of the clean data: (400, 10)


### 📊 Step 15: Post-Processing Data Quality & Boundary Verification

In [19]:
print("--- Missing Values Count in Final Data ---")
print(df_winsorized.isnull().sum())

print("\n--- Basic Statistics of Numerical Columns ---")
print(df_winsorized[['bmi', 'blood_pressure', 'cholesterol', 'glucose']].describe().loc[['min', 'max']])

--- Missing Values Count in Final Data ---
patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
age_isna          0
dtype: int64

--- Basic Statistics of Numerical Columns ---
       bmi  blood_pressure  cholesterol  glucose
min  19.22           97.56   162.010000    76.72
max  34.85          176.18   246.123333   220.78


### 📝 Part C: Project Brief Report

#### 1. Which imputation strategy was most effective?
* **Answer:** For numerical columns like `bmi`, `cholesterol`, and `glucose`, advanced methods like **MICE (IterativeImputer)** and **KNN Imputer** were the most effective because they capture relationships between different health attributes instead of just filling a simple average. For categorical columns (`gender`, `region`), **Most Frequent Imputation** worked perfectly to maintain class distribution.

#### 2. Which outlier handling method preserved data quality best?
* **Answer:** **Winsorization (Capping)** preserved data quality the best. Methods like Z-score and IQR completely delete rows, which causes data loss (reducing our 400 rows). Winsorization keeps all 400 records intact by adjusting extreme outlier values to the 5th and 95th percentiles, ensuring we do not lose patient data.

#### 3. How data cleaning improved dataset usability?
* **Answer:** Initially, the dataset had missing entries and extreme unrealistic spikes (outliers) which would make machine learning models fail or give wrong predictions. Data cleaning resolved all issues, brought missing counts to 0, stabilized data variance, and created a **100% complete, machine learning-ready dataset** to predict `disease_risk` accurately.